# 01 · Match the answer to the decision

**You will:** choose between categories, ordered levels and yes/no probabilities; inspect the full answer; improve a question.

| Type | Example | How code uses it |
|---|---|---|
| Choice | Which support team? | Route to one branch |
| Score | How frustrated is this customer? | Compare a position on a rubric |
| Noul | Are they asking for money back? | Threshold a probability |

Choice options are unordered. Score levels are ordered, and its returned average can lie between them. Noul 0.5 means uncertain truth, not medium intensity. We will later map these to OpenAI Decisions' choice, score and predicate.

In [ ]:
# Find the course folder and load API settings without printing secret keys.
from pathlib import Path
import sys
import json
from IPython.display import display
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / 'course_utils').is_dir():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open Jupyter inside the course folder.')
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from course_utils.config import load_settings, require_key
settings = load_settings(ROOT)
print('Python:', sys.version.split()[0], '| Mode:', settings.mode)
print('Jev key configured:', bool(settings.typesafe_api_key))
print('OpenAI key configured:', bool(settings.openai_api_key))

In [ ]:
# Load the course tickets; reference labels are for evaluation only.
tickets = []
for line in (ROOT / 'data/tickets.jsonl').read_text().splitlines():
    tickets.append(json.loads(line))

In [ ]:
# Define the categories, ordered levels, and refund question.
TEAMS = {
    'billing': 'Existing payments, refunds, invoices, and subscriptions.',
    'technical': 'Malfunctioning application features or service errors.',
    'account': 'Login, credentials, profiles, and membership.',
    'other': 'Sales, suggestions, general questions, or insufficient evidence.',
}
FRUSTRATION = [
    'Neutral or positive.',
    'Mild complaint or annoyance.',
    'Clear or repeated frustration.',
    'Intense anger or explicit outrage.',
]
TEAM_PROMPT = 'Which team should handle the ticket? For mixed intents choose the first actionable issue.'
REFUND_PROMPT = 'Does the customer explicitly request money back? Policy questions, cancellation alone, and negated requests are not requests.'
from typesafe_sdk import TypeSafeClient, Choice, Score, Noul, RetryPolicy
questions = {
    'team': Choice(instructions=TEAM_PROMPT, criteria=TEAMS),
    'frustration': Score(
        instructions='How frustrated is the customer, based on expressed tone rather than urgency?',
        criteria=FRUSTRATION,
    ),
    'refund_requested': Noul(instructions=REFUND_PROMPT),
}

In [ ]:
# Select one ticket and show exactly what the model will receive.
example_id = 'dev-refund'
ticket = next(ticket for ticket in tickets if ticket['id'] == example_id)
state = {'ticket': ticket['text']}
print('State analysed:')
display(state)

In [ ]:
# Ask all three independent questions in one Jev request.
response = None
if settings.mode != 'live' or not settings.typesafe_api_key:
    print('Set TYPESAFE_API_KEY and COURSE_MODE=live, then rerun setup.')
else:
    with TypeSafeClient(
        api_key=require_key(settings, 'typesafe'),
        retry=RetryPolicy(max_retries=1),
        timeout=30,
    ) as client:
        response = client.system_one(model=settings.jev_new, state=state, questions=questions)

## Read the distribution, not just the winner

First guess which team should handle the ticket. Then compare your guess with the chart.

`other` catches inputs outside the named teams. It does not catch every uncertain answer: you still need a rule for human review. All three questions read the same ticket independently; they do not use each other's answers.

In [ ]:
# Show the ticket and plot the probability assigned to each team.
import matplotlib.pyplot as plt

if response is None:
    print('No response yet. Run the request cell first.')
else:
    print('State analysed:')
    display(state)
    team_answer = response.choices['team']
    figure, axes = plt.subplots(figsize=(7, 3))
    axes.bar(team_answer.probabilities.keys(), team_answer.probabilities.values())
    axes.set(ylim=(0, 1), ylabel='Probability', title='Team distribution')
    plt.show()
    print('Selected team:', team_answer.choice)
    print('Confidence statistic:', team_answer.confidence)

In [ ]:
# Read the ordered frustration score and the separate refund probability.
if response is not None:
    display(state)
    frustration_answer = response.scores['frustration']
    print('Frustration score:', frustration_answer.score)
    print('Level descriptions:', frustration_answer.legend)
    print('Refund probability:', response.nouls['refund_requested'].noul)

## Probability and confidence answer different questions

TypeSafe's Choice confidence rescales the winning probability relative to a uniform distribution. Score confidence instead accounts for how far probability mass lies from the most likely rubric level. Adjacent-level uncertainty differs from uncertainty between opposite extremes. Neither statistic proves the prediction is correct. Noul has no separate confidence field.

For example, a hypothetical three-way Choice with probabilities 0.6/0.3/0.1 has confidence 0.4. These are arithmetic illustrations, not API results. Examine the [published formulas](https://docs.typesafe.ai/confidence) and validate thresholds using your own labelled cases.

## Exercise

Replace “Is this customer unhappy?” with a question that measures **frustration**, and another that detects **a refund request**. Run `dev-policy`, `dev-negation`, and `dev-mixed`. Explain why the type differs. Expected behavior: policy/negation are not refund requests; mixed intent needs review even if a team is confidently selected.

## Solution

Use Score with the four explicit tone levels for frustration, and Noul with the refund definition already provided. Inspect mixed intent manually rather than assuming the selected team addresses every problem. The cell below lets you try one development case at a time.

In [ ]:
# Try one exercise ticket at a time; change the ID and rerun this cell.
exercise_id = 'dev-negation'  # Also try 'dev-policy' and 'dev-mixed'.
exercise_ticket = next(ticket for ticket in tickets if ticket['id'] == exercise_id)
exercise_state = {'ticket': exercise_ticket['text']}
print('State analysed:')
display(exercise_state)

if settings.mode == 'live' and settings.typesafe_api_key:
    with TypeSafeClient(
        api_key=require_key(settings, 'typesafe'),
        retry=RetryPolicy(max_retries=1),
        timeout=30,
    ) as client:
        exercise_response = client.system_one(
            model=settings.jev_new, state=exercise_state, questions=questions,
        )
    print('Refund probability:', exercise_response.nouls['refund_requested'].noul)
    print('Frustration score:', exercise_response.scores['frustration'].score)

## What would a different approach do?

A conventional classifier learns a fixed label mapping from training examples; that can be excellent when labels and distributions are stable. A small LLM or reasoning model can generate answers but has different latency and cost trade-offs. JSON mode targets parseable JSON; schema-constrained Structured Outputs restrict the shape further. Neither formatting mechanism establishes semantic accuracy. Use deterministic code for exact matches and arithmetic.

Next: combine these judgments into a policy, keeping business rules outside the model.

Sources: [question types](https://docs.typesafe.ai/primitives), [confidence](https://docs.typesafe.ai/confidence).